## 1. Imports

In [0]:
import os
import zipfile
import requests
import time
from pyspark.sql import functions as f
from pyspark.sql import Window

## 2. Directorios y defincicion de URLs

In [0]:
BASE = "/Volumes/mine4213/proyecto/data"

ZIP_DIR = f"{BASE}/zip"
CSV_DIR = f"{BASE}/csv"

print("BASE:", BASE)
print("ZIP_DIR:", ZIP_DIR)
print("CSV_DIR:", CSV_DIR)

In [0]:
os.makedirs(ZIP_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

print("Directorios listos.")

In [0]:
URLS = [
    f"https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_{day:02d}.zip"
    for day in range(1, 8)
]

for url in URLS:
    print(url)

## 3. Descarga archivos

In [0]:
def validate_zip(zip_path, expected_csv_name):
    """
    Valida la integridad básica de un archivo ZIP.

    Comprueba:
    - existencia
    - tamaño > 0
    - estructura ZIP válida
    - CRC de los archivos internos
    - presencia del CSV esperado
    """

    if not os.path.exists(zip_path):
        return False, "El archivo no existe"

    if os.path.getsize(zip_path) == 0:
        return False, "El archivo está vacío"

    if not zipfile.is_zipfile(zip_path):
        return False, "El archivo no es un ZIP válido"

    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            bad_file = z.testzip()

            if bad_file is not None:
                return False, f"Error CRC en {bad_file}"

            files = z.namelist()

            if expected_csv_name not in files:
                return False, f"No contiene {expected_csv_name}"

    except Exception as e:
        return False, str(e)

    return True, "OK"

In [0]:
def download_with_retries(url, destination, max_retries=3, timeout=120):
    """
    Descarga un archivo con reintentos.

    Si el archivo ya existe y es válido,
    la descarga puede omitirse posteriormente.
    """

    for attempt in range(1, max_retries + 1):

        try:
            print(f"Intento {attempt}/{max_retries}: {url}")

            with requests.get(
                url,
                stream=True,
                timeout=timeout
            ) as response:

                response.raise_for_status()

                with open(destination, "wb") as f:

                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            f.write(chunk)

            print(
                f"Descargado: {os.path.basename(destination)} "
                f"({os.path.getsize(destination) / 1024**2:.2f} MB)"
            )

            return

        except Exception as e:

            print(f"Error: {e}")

            if attempt == max_retries:
                raise

            wait_seconds = attempt * 5

            print(f"Reintentando en {wait_seconds} segundos...")
            time.sleep(wait_seconds)

In [0]:
download_results = []

for day, url in enumerate(URLS, start=1):

    zip_name = f"AIS_2023_06_{day:02d}.zip"
    csv_name = f"AIS_2023_06_{day:02d}.csv"

    zip_path = f"{ZIP_DIR}/{zip_name}"

    # Si ya existe, primero verificamos si es válido
    valid, message = validate_zip(
        zip_path,
        csv_name
    )

    if valid:
        print(f"{zip_name}: ya existe y es válido.")

    else:
        print(f"{zip_name}: debe descargarse. Motivo: {message}")

        download_with_retries(
            url,
            zip_path
        )

        # Verificar después de descargar
        valid, message = validate_zip(
            zip_path,
            csv_name
        )

        if not valid:
            raise RuntimeError(
                f"Falló la verificación de {zip_name}: {message}"
            )

    download_results.append(
        (
            zip_name,
            os.path.getsize(zip_path),
            valid,
            message
        )
    )

In [0]:
download_df = spark.createDataFrame(
    download_results,
    [
        "archivo",
        "bytes",
        "integridad_ok",
        "resultado_validacion"
    ]
)

display(download_df)

In [0]:
for day in range(1, 8):

    zip_name = f"AIS_2023_06_{day:02d}.zip"
    csv_name = f"AIS_2023_06_{day:02d}.csv"

    zip_path = f"{ZIP_DIR}/{zip_name}"
    csv_path = f"{CSV_DIR}/{csv_name}"

    if os.path.exists(csv_path) and os.path.getsize(csv_path) > 0:
        print(f"{csv_name}: ya existe.")

    else:
        print(f"Extrayendo {zip_name}...")

        with zipfile.ZipFile(zip_path, "r") as z:
            z.extract(csv_name, CSV_DIR)

        print(f"Extraído: {csv_name}")

In [0]:
csv_files = sorted([
    f
    for f in os.listdir(CSV_DIR)
    if f.startswith("AIS_2023_06_") and f.endswith(".csv")
])

print(f"Cantidad de CSV encontrados: {len(csv_files)}")

for fname in csv_files:
    print(fname)

In [0]:
expected_files = {
    f"AIS_2023_06_{day:02d}.csv"
    for day in range(1, 8)
}

actual_files = set(csv_files)

missing_files = expected_files - actual_files
unexpected_files = actual_files - expected_files

print("Faltantes:", missing_files)
print("No esperados:", unexpected_files)

assert len(missing_files) == 0, (
    f"Faltan archivos: {missing_files}"
)

assert len(actual_files) == 7, (
    f"Se esperaban 7 archivos y hay {len(actual_files)}"
)

## 4. Esquema

In [0]:
headers = {}

for file_name in csv_files:

    file_path = f"{CSV_DIR}/{file_name}"

    with open(file_path, "r", encoding="utf-8") as file:
        header = file.readline().strip()

    headers[file_name] = header

    print(file_name)
    print(header)
    print("-" * 100)

unique_headers = set(headers.values())

print(
    "Cantidad de encabezados diferentes:",
    len(unique_headers)
)

## 5. Carga

In [0]:
AIS_SCHEMA = """
    MMSI string,
    BaseDateTime timestamp,
    LAT double,
    LON double,
    SOG float,
    COG float,
    Heading float,
    VesselName string,
    IMO string,
    CallSign string,
    VesselType smallint,
    Status smallint,
    Length float,
    Width float,
    Draft float,
    Cargo string,
    TransceiverClass string
"""

ais_raw = (
    spark.read
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(AIS_SCHEMA)
    .csv(f"{CSV_DIR}/AIS_2023_06_*.csv")
)

ais_raw.printSchema()

display(ais_raw.limit(20))

## 6. Validaciones

In [0]:
ais_raw = (
    ais_raw
    .withColumn(
        "fecha",
        f.to_date("BaseDateTime")
    )
    .withColumn(
        "archivo_origen",
        f.regexp_extract(
            f.col("_metadata.file_name"),
            r"(AIS_2023_06_\d{2}\.csv)",
            1
        )
    )
)

In [0]:
input_files = sorted(
    [row["file_path"] for row in
     spark.read.csv(f"{CSV_DIR}/AIS_2023_06_*.csv", header=True)
     .select("_metadata.file_path")
     .distinct()
     .collect()]
)

print(
    f"Spark detectó {len(input_files)} archivos."
)

for path in input_files:
    print(path)

assert len(input_files) == 7, (
    f"Spark debería leer 7 archivos, pero detectó {len(input_files)}."
)

In [0]:
expected_dates = {
    "2023-06-01",
    "2023-06-02",
    "2023-06-03",
    "2023-06-04",
    "2023-06-05",
    "2023-06-06",
    "2023-06-07"
}

actual_dates = {
    str(row["fecha"])
    for row in (
        ais_raw
        .select("fecha")
        .distinct()
        .collect()
    )
}

print("Fechas encontradas:", actual_dates)

assert actual_dates == expected_dates